In [8]:
using JuMP, Ipopt
using LinearAlgebra

In [6]:
# Day Ahead Market

P = 2 * [0.11  0      0;
         0     0.085  0;
         0     0      0.1225]

Q = [5, 1.2, 1]
A = [0, 0, 0]
B = [250, 300, 270]
G = [1, 1, 1]

h = 600

model = Model(Ipopt.Optimizer)

@variable(model, x[1:3])

@constraint(model, A .<= x .<= B)
@constraint(model, G' * x == h)

@objective(model, Min, (1/2) * x' * P * x + Q' * x)

set_silent(model)
optimize!(model)

println("\nDay Ahead Market Optimal Dispatch")
println("---------------------------------")

for i in 1:3
    println("Generator $i: $(round(value(x[i]), digits=2)) MW")
end

println("---------------------------------")
println("Total Dispatch: $(round(sum(value.(x)), digits=2)) MW")
println("Total Cost: \$$(round(objective_value(model), digits=2))")


Day Ahead Market Optimal Dispatch
---------------------------------
Generator 1: 175.85 MW
Generator 2: 249.92 MW
Generator 3: 174.23 MW
---------------------------------
Total Dispatch: 600.0 MW
Total Cost: $13782.66


In [9]:
# Real Time Market

p_DA = value.(x[1:3])  # Save DA dispatch before redefining x
Pg_RT = 620.0

kup = 1.0
kdown = 1.0

a = 2 * [0.11, 0.085, 0.1225]
c = [5.0, 1.2, 1.0]

P = [Diagonal(a)   -1 .* Diagonal(a);
    -1 .* Diagonal(a)   Diagonal(a)]

Q = vcat(a .* p_DA .+ c .+ kup, a .* p_DA .- c .+ kdown)

model = Model(Ipopt.Optimizer)
@variable(model, x[1:6] >= 0)
@constraint(model, [i=1:3], x[i] <= B[i] - p_DA[i])
@constraint(model, [i=1:3], x[i+3] <= p_DA[i])
@constraint(model,sum(x[1:3]) - sum(x[4:6]) == Pg_RT - sum(p_DA))
@objective(model, Min,(1/2) * x' * P * x + Q' * x)

set_silent(model)
optimize!(model)

# Display results
println("\nReal Time Market Optimal Dispatch")
println("---------------------------------")

for i in 1:3
    println("Generator $i:")
    println("  Upward Adjustment:   $(round(value(x[i]), digits=2)) MW")
    println("  Downward Adjustment: $(round(value(x[i+3]), digits=2)) MW")
    println("  Real-Time Output:    $(round(p_DA[i] + value(x[i]) - value(x[i+3]), digits=2)) MW")
end

println("---------------------------------")
println("Total Real-Time Dispatch: $(round(sum(p_DA) + sum(value.(x[1:3])) - sum(value.(x[4:6])), digits=2)) MW")
println("Total Cost: \$$(round(objective_value(model), digits=2))")


Real Time Market Optimal Dispatch
---------------------------------
Generator 1:
  Upward Adjustment:   6.27 MW
  Downward Adjustment: -0.0 MW
  Real-Time Output:    182.11 MW
Generator 2:
  Upward Adjustment:   8.11 MW
  Downward Adjustment: -0.0 MW
  Real-Time Output:    258.03 MW
Generator 3:
  Upward Adjustment:   5.63 MW
  Downward Adjustment: -0.0 MW
  Real-Time Output:    179.86 MW
---------------------------------
Total Real-Time Dispatch: 620.0 MW
Total Cost: $907.52


In [10]:
function day_ahead_dispatch(h)

    P = 2 * [0.11  0      0;
             0     0.085  0;
             0     0      0.1225]

    Q = [5, 1.2, 1]
    A = [0, 0, 0]
    B = [250, 300, 270]
    G = [1, 1, 1]

    model = Model(Ipopt.Optimizer)

    @variable(model, x[1:3])

    @constraint(model, A .<= x .<= B)
    @constraint(model, G' * x == h)

    @objective(model, Min, (1/2) * x' * P * x + Q' * x)

    set_silent(model)
    optimize!(model)

    generation = value.(x)
    total_cost = objective_value(model)

    return generation, total_cost
end

day_ahead_dispatch (generic function with 1 method)

In [ ]:
function real_time_dispatch(p_DA, Pg_RT, kup = 1.0, kdown = 1.0)

    B = [250, 300, 270]

    a = 2 * [0.11, 0.085, 0.1225]
    c = [5.0, 1.2, 1.0]

    P = [Diagonal(a)   -1 .* Diagonal(a);
        -1 .* Diagonal(a)   Diagonal(a)]

    Q = vcat(a .* p_DA .+ c .+ kup, a .* p_DA .- c .+ kdown)

    model = Model(Ipopt.Optimizer)

    @variable(model, x[1:6] >= 0)
    @constraint(model, [i=1:3], x[i] <= B[i] - p_DA[i])
    @constraint(model, [i=1:3], x[i+3] <= p_DA[i])
    @constraint(model, sum(x[1:3]) - sum(x[4:6]) == Pg_RT - sum(p_DA))

    @objective(model, Min, (1/2) * x' * P * x + Q' * x)

    set_silent(model)
    optimize!(model)

    upward = value.(x[1:3])
    downward = value.(x[4:6])
    total_cost = objective_value(model)

    return upward, downward, total_cost
end

real_time_dispatch (generic function with 1 method)

In [3]:
include("market_functions.jl")
day_ahead_dispatch(575)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************



([168.01617149306307, 239.78563369681387, 167.19819481012303], 12712.033753290709)